Train or test split, preprocessing and SMOTE

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)
from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt

# assume df_fraud already has:
# - class
# - purchase_value, age
# - time_since_signup_hours
# - user_tx_count_total, tx_count_24h, tx_count_7d
# - source, browser, sex, ip_country

target_col = "class"

num_cols = [
    "purchase_value",
    "age",
    "time_since_signup_hours",
    "user_tx_count_total",
    "tx_count_24h",
    "tx_count_7d",
]

cat_cols = [
    "source",
    "browser",
    "sex",
    "ip_country",
]

X = df_fraud[num_cols + cat_cols].copy()
y = df_fraud[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

# fit preprocessor on train only
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Class counts before SMOTE:", np.bincount(y_train))

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_prepared, y_train)

print("Class counts after SMOTE: ", np.bincount(y_resampled))


Task 2: Baseline models on Fraud_Data

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n=== {name} ===")
    print("ROC AUC:", roc_auc_score(y_test, y_prob))

    print("\nClassification report (threshold 0.5):")
    print(classification_report(y_test, y_pred))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    RocCurveDisplay.from_predictions(y_test, y_prob)
    plt.title(f"ROC curve - {name}")
    plt.show()

# Logistic Regression baseline
log_reg = LogisticRegression(
    max_iter=500,
    class_weight=None,
    n_jobs=-1,
)

log_reg.fit(X_resampled, y_resampled)

evaluate_model("Logistic Regression (Fraud_Data)", log_reg, X_test_prepared, y_test)

# RandomForest baseline
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_resampled, y_resampled)

evaluate_model("RandomForest (Fraud_Data)", rf, X_test_prepared, y_test)


Task 3: Hyperparameter tuning and best model

In [ ]:
from sklearn.model_selection import GridSearchCV

# search for Logistic Regression
log_reg_base = LogisticRegression(max_iter=500, n_jobs=-1)

log_reg_param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "class_weight": [None, "balanced"],
}

log_reg_grid = GridSearchCV(
    estimator=log_reg_base,
    param_grid=log_reg_param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

log_reg_grid.fit(X_resampled, y_resampled)

print("Best Logistic Regression params:", log_reg_grid.best_params_)
best_log_reg = log_reg_grid.best_estimator_

evaluate_model("Logistic Regression tuned (Fraud_Data)", best_log_reg, X_test_prepared, y_test)

# search for RandomForest
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
}

rf_grid = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

rf_grid.fit(X_resampled, y_resampled)

print("Best RandomForest params:", rf_grid.best_params_)
best_rf = rf_grid.best_estimator_

evaluate_model("RandomForest tuned (Fraud_Data)", best_rf, X_test_prepared, y_test)


In [ ]:
import joblib
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True, parents=True)

joblib.dump(
    {
        "preprocessor": preprocessor,
        "model": best_rf,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
    },
    models_dir / "fraud_best_model.joblib",
)

print("Saved model to", models_dir / "fraud_best_model.joblib")


Task 3: SHAP explainability for Fraud_Data

In [ ]:
import shap

# shap needs dense array
X_test_dense = X_test_prepared.toarray() if hasattr(X_test_prepared, "toarray") else X_test_prepared

# use a subset for speed
X_test_sample = X_test_dense[:500]

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test_sample)

# we need feature names from the preprocessor
# numeric cols first, then one hot encoded categorical feature names
ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)

feature_names = np.concatenate([num_cols, cat_feature_names])

shap.summary_plot(shap_values[1], X_test_sample, feature_names=feature_names)


In [ ]:
idx = 0  # pick a test sample index
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][idx],
    X_test_sample[idx],
    feature_names=feature_names,
    matplotlib=True,
)


Split and scaling

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

# df_cc is the creditcard dataframe

feature_cols = [c for c in df_cc.columns if c != "Class"]
target_col = "Class"

X = df_cc[feature_cols].copy()
y = df_cc[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Class counts before SMOTE:", np.bincount(y_train))

smote = SMOTE(random_state=42)
X_resampled_cc, y_resampled_cc = smote.fit_resample(X_train_scaled, y_train)

print("Class counts after SMOTE:", np.bincount(y_resampled_cc))


Baseline models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def evaluate_model_cc(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n=== {name} ===")
    print("ROC AUC:", roc_auc_score(y_test, y_prob))

    print("\nClassification report (threshold 0.5):")
    print(classification_report(y_test, y_pred))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    RocCurveDisplay.from_predictions(y_test, y_prob)
    plt.title(f"ROC curve - {name}")
    plt.show()

log_reg_cc = LogisticRegression(
    max_iter=500,
    class_weight=None,
    n_jobs=-1,
)

log_reg_cc.fit(X_resampled_cc, y_resampled_cc)
evaluate_model_cc("Logistic Regression (creditcard)", log_reg_cc, X_test_scaled, y_test)

rf_cc = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)

rf_cc.fit(X_resampled_cc, y_resampled_cc)
evaluate_model_cc("RandomForest (creditcard)", rf_cc, X_test_scaled, y_test)


Task 4: API, Docker and CI

In [ ]:
from pathlib import Path
from typing import List, Optional, Dict, Any

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

# load artifacts
models_dir = Path(__file__).resolve().parents[2] / "models"
artifact_path = models_dir / "fraud_best_model.joblib"

artifact = joblib.load(artifact_path)

preprocessor = artifact["preprocessor"]
model = artifact["model"]
num_cols = artifact["num_cols"]
cat_cols = artifact["cat_cols"]

app = FastAPI(title="Fraud Detection API", version="1.0.0")

class FraudRequest(BaseModel):
    features: Dict[str, Any]

class FraudResponse(BaseModel):
    fraud_probability: float
    is_fraud: int

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=FraudResponse)
def predict(req: FraudRequest):
    # convert dict to DataFrame of one row
    df = pd.DataFrame([req.features])

    # ensure all expected columns exist
    for col in num_cols + cat_cols:
        if col not in df.columns:
            df[col] = np.nan

    df = df[num_cols + cat_cols]

    X_prepared = preprocessor.transform(df)
    prob = float(model.predict_proba(X_prepared)[:, 1][0])
    label = int(prob >= 0.5)

    return FraudResponse(
        fraud_probability=prob,
        is_fraud=label,
    )


CI workflow

In [ ]:
name: CI

on:
  push:
  pull_request:

jobs:
  tests:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout code
      uses: actions/checkout@v4

    - name: Set up Python
      uses: actions/setup-python@v5
      with:
        python-version: "3.11"

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt

    - name: Run tests
      run: |
        pytest
